# 13. Historical Value at Risk Report

## Objective

This notebook demonstrates how Historical Value at Risk (VaR) is calculated from a distribution of historical profit and loss (P&L).

Workflow

Historical Scenarios

↓

Historical VaR Engine

↓

Historical P&L Distribution

↓

Historical VaR Report

The report summarizes the historical P&L distribution using common market risk statistics, including the 95% and 99% Historical VaR.

In [1]:
from datetime import date

from src.common.enums import (
    DayCountConvention,
    FloatingIndex,
    PayReceive,
    PaymentFrequency,
)

from src.curves.bootstrapper import Bootstrapper

from src.market_data.market_instrument import MarketInstrument
from src.market_data.market_quote_collection import (
    MarketQuoteCollection,
)

from src.products.trade import Trade
from src.products.fixed_leg import FixedLeg
from src.products.floating_leg import FloatingLeg
from src.products.interest_rate_swap import (
    InterestRateSwap,
)

from src.risk.historical_scenario import HistoricalScenario
from src.risk.historical_var_engine import HistoricalVaREngine
from src.risk.historical_var_report_engine import (
    HistoricalVaRReportEngine,
)

## Build Yield Curve

In [2]:
quotes = MarketQuoteCollection()

quotes.add(MarketInstrument("Deposit", "1M", 0.05))
quotes.add(MarketInstrument("Deposit", "3M", 0.05))
quotes.add(MarketInstrument("Deposit", "6M", 0.05))
quotes.add(MarketInstrument("Deposit", "1Y", 0.05))
quotes.add(MarketInstrument("Deposit", "2Y", 0.05))
quotes.add(MarketInstrument("Deposit", "3Y", 0.05))

curve = Bootstrapper(
    valuation_date=date(2026, 1, 1),
    market_quotes=quotes,
).build()

## Build Interest Rate Swap

In [3]:
trade = Trade(
    trade_id="IRS001",
    counterparty="ABC",
    currency="USD",
    effective_date=date(2026, 1, 1),
    maturity_date=date(2029, 1, 1),
)

fixed_leg = FixedLeg(
    notional=1_000_000,
    fixed_rate=0.05,
    payment_frequency=PaymentFrequency.ANNUAL,
    day_count=DayCountConvention.ACT_365,
    pay_receive=PayReceive.RECEIVE,
)

floating_leg = FloatingLeg(
    notional=1_000_000,
    payment_frequency=PaymentFrequency.ANNUAL,
    day_count=DayCountConvention.ACT_365,
    pay_receive=PayReceive.PAY,
    floating_index=FloatingIndex.SOFR,
)

swap = InterestRateSwap(
    trade=trade,
    fixed_leg=fixed_leg,
    floating_leg=floating_leg,
)

## Create Historical Market Scenarios

In [4]:
historical_scenarios = [

    HistoricalScenario(
        scenario_date=date(2025, 1, 14),
        tenor_shifts={
            "1Y": 0.0003,
            "2Y": 0.0005,
            "3Y": 0.0007,
        },
    ),

    HistoricalScenario(
        scenario_date=date(2025, 1, 15),
        tenor_shifts={
            "1Y": -0.0002,
            "2Y": -0.0004,
            "3Y": -0.0005,
        },
    ),

    HistoricalScenario(
        scenario_date=date(2025, 1, 16),
        tenor_shifts={
            "1Y": 0.0008,
            "2Y": 0.0006,
            "3Y": 0.0002,
        },
    ),

]

## Generate Historical P&L Distribution

In [5]:
scenario_results = HistoricalVaREngine().run(
    swap,
    curve,
    historical_scenarios,
)

## Calculate Historical VaR Statistics

In [6]:
report = HistoricalVaRReportEngine().run(
    scenario_results
)

In [7]:
print("Historical VaR Summary")
print("-" * 40)

print(
    f"Number of Scenarios : {report.number_of_scenarios}"
)

print(
    f"Best Gain           : {report.best_gain:,.2f}"
)

print(
    f"Worst Loss          : {report.worst_loss:,.2f}"
)

print(
    f"Average P&L         : {report.average_pnl:,.2f}"
)

print(
    f"Median P&L          : {report.median_pnl:,.2f}"
)

print(
    f"95% Historical VaR  : {report.var_95:,.2f}"
)

print(
    f"99% Historical VaR  : {report.var_99:,.2f}"
)

Historical VaR Summary
----------------------------------------
Number of Scenarios : 3
Best Gain           : 110.50
Worst Loss          : -150.01
Average P&L         : -52.56
Median P&L          : -118.18
95% Historical VaR  : 150.01
99% Historical VaR  : 150.01


## Interpretation

Historical Value at Risk estimates potential future losses by replaying historical market movements on today's portfolio.

The workflow is:

1. Apply each historical yield curve movement.
2. Reprice the swap.
3. Record the resulting profit or loss.
4. Construct the historical P&L distribution.
5. Calculate summary statistics, including the 95% and 99% Historical VaR.

Unlike DV01, which measures local sensitivity, Historical VaR captures the impact of realistic historical market movements across the entire yield curve.